In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!ls /content/drive/MyDrive/ColabNotebooks

00_pytorch_fundamantals.ipynb  labels  Untitled0.ipynb
disparity		       rgb     Untitled1.ipynb


In [8]:
!pip install opencv-python

In [4]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Set your paths (change this based on your Drive)
base_path = '/content/drive/MyDrive/ColabNotebooks'
rgb_dir = os.path.join(base_path, 'rgb')
disp_dir = os.path.join(base_path, 'disparity')
labels_dir = os.path.join(base_path, 'labels')
rgbd_dir = os.path.join(base_path, 'rgbd')  # Output 4-channel images
os.makedirs(rgbd_dir, exist_ok=True)

# Process each image
for fname in tqdm(sorted(os.listdir(rgb_dir))):
    if not fname.endswith('.jpg'):
        continue

    name = os.path.splitext(fname)[0]
    rgb_path = os.path.join(rgb_dir, f"{name}.jpg")
    disp_path = os.path.join(disp_dir, f"{name}.png")
    label_path = os.path.join(labels_dir, f"{name}.txt")

    # Load images
    rgb = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
    disp = cv2.imread(disp_path, cv2.IMREAD_GRAYSCALE)

    if rgb is None or disp is None:
        print(f"❌ Missing RGB or disparity for {name}")
        continue

    # Normalize disparity if needed
    disp = cv2.normalize(disp, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Stack as 4-channel
    rgbd = cv2.merge([rgb[..., 0], rgb[..., 1], rgb[..., 2], disp])

    # Save as PNG
    out_path = os.path.join(rgbd_dir, f"{name}.png")
    cv2.imwrite(out_path, rgbd)

print("✅ Fused RGBD images saved.")


100%|██████████| 505/505 [03:54<00:00,  2.16it/s]

✅ Fused RGBD images saved.


In [7]:
import os
import random

# Set seed for reproducibility
random.seed(42)

# Define paths
rgbd_dir = '/content/drive/MyDrive/ColabNotebooks/rgbd'
output_dir = '/content/drive/MyDrive/ColabNotebooks'
ext = '.png'  # or '.jpg' if you saved RGBD as jpg

# Get all RGBD image paths
all_files = [f for f in os.listdir(rgbd_dir) if f.endswith(ext)]
all_files.sort()
random.shuffle(all_files)

# Split into train/val/test
train_split = 0.7
val_split = 0.2
test_split = 0.1

n = len(all_files)
n_train = int(n * train_split)
n_val = int(n * val_split)

train_files = all_files[:n_train]
val_files = all_files[n_train:n_train + n_val]
test_files = all_files[n_train + n_val:]

def write_list(file_list, filename):
    with open(os.path.join(output_dir, filename), 'w') as f:
        for fname in file_list:
            f.write(os.path.join(rgbd_dir, fname) + '\n')

# Write out the .txt files
write_list(train_files, 'train.txt')
write_list(val_files, 'val.txt')
write_list(test_files, 'test.txt')

print("✅ train.txt, val.txt, and test.txt written successfully.")



✅ train.txt, val.txt, and test.txt written successfully.


In [8]:
from torch.utils.data import Dataset
from PIL import Image
import os
import torch

class RGBDDataset(Dataset):
    def __init__(self, file_list, label_dir, transform=None):
        self.file_list = file_list
        self.label_dir = label_dir
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        rgbd_path = self.file_list[idx]
        base = os.path.splitext(os.path.basename(rgbd_path))[0]
        label_path = os.path.join(self.label_dir, base + '.txt')

        # Load RGBD image (assumes 4-channel PNG)
        image = Image.open(rgbd_path).convert('RGBA')  # 4-channel
        if self.transform:
            image = self.transform(image)

        # Load YOLO-format labels
        boxes = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    cls, x_center, y_center, width, height = map(float, parts)
                    boxes.append([cls, x_center, y_center, width, height])
        boxes = torch.tensor(boxes)

        return image, boxes


In [9]:
from torch.utils.data import DataLoader
from torchvision import transforms

# Read txt files
def load_split(path):
    with open(path) as f:
        return [line.strip() for line in f.readlines()]

train_list = load_split('/content/drive/MyDrive/ColabNotebooks/train.txt')
val_list = load_split('/content/drive/MyDrive/ColabNotebooks/val.txt')

transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),  # Converts RGBA to 4xHxW
])

train_dataset = RGBDDataset(train_list, '/content/drive/MyDrive/ColabNotebooks/labels', transform=transform)
val_dataset = RGBDDataset(val_list, '/content/drive/MyDrive/ColabNotebooks/labels', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

print("✅ Data loaders ready.")


✅ Data loaders ready.


In [10]:
!pip install ultralytics --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.1 MB/s eta 0:00:00


In [11]:
import torch
import torch.nn as nn
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO('yolov8n.pt')  # Or yolov8s.pt etc.

# Patch the first layer to accept 4-channel input
conv1 = model.model.model[0]
if isinstance(conv1, nn.Conv2d) and conv1.in_channels == 3:
    print("Patching first layer from 3 to 4 channels...")
    new_conv = nn.Conv2d(4, conv1.out_channels, conv1.kernel_size, conv1.stride, conv1.padding, bias=conv1.bias is not None)
    with torch.no_grad():
        new_conv.weight[:, :3] = conv1.weight  # copy RGB weights
        new_conv.weight[:, 3] = conv1.weight[:, 0]  # init D channel like R
        if conv1.bias is not None:
            new_conv.bias = conv1.bias
    model.model.model[0] = new_conv
else:
    print("⚠️ First layer is already patched or unexpected structure.")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
⚠️ First layer is already patched or unexpected structure.


In [12]:
from ultralytics import YOLO
import torch.nn as nn
import torch

# Load pretrained model
model = YOLO('yolov8n.pt')

# Access the first Conv layer (Ultralytics uses a custom Conv class)
first_layer = model.model.model[0]

# Extract original parameters
out_channels = first_layer.conv.out_channels
kernel_size = first_layer.conv.kernel_size
stride = first_layer.conv.stride
padding = first_layer.conv.padding
bias = first_layer.conv.bias is not None

# Replace with a standard Conv2d layer accepting 4 channels
first_layer.conv = nn.Conv2d(
    in_channels=4,
    out_channels=out_channels,
    kernel_size=kernel_size,
    stride=stride,
    padding=padding,
    bias=bias
)

print("✅ Successfully patched YOLOv8 to accept 4-channel RGBD images.")



✅ Successfully patched YOLOv8 to accept 4-channel RGBD images.


In [14]:
import os
import shutil
import random
from tqdm import tqdm

# Set paths
base_path = '/content/drive/MyDrive/ColabNotebooks'
rgbd_dir = os.path.join(base_path, 'rgbd')
labels_dir = os.path.join(base_path, 'labels')  # Original label folder

# Output folders
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(base_path, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(base_path, split, 'labels'), exist_ok=True)

# List all RGBD .png files
all_images = [f for f in os.listdir(rgbd_dir) if f.endswith('.png')]
all_images.sort()
random.shuffle(all_images)

# Split ratios
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

total = len(all_images)
train_count = int(train_ratio * total)
val_count = int(val_ratio * total)

train_files = all_images[:train_count]
val_files = all_images[train_count:train_count + val_count]
test_files = all_images[train_count + val_count:]

# Function to copy image + label
def copy_files(file_list, split_name):
    for img_file in tqdm(file_list, desc=f"Copying to {split_name}"):
        name = os.path.splitext(img_file)[0]
        label_file = name + '.txt'

        src_img = os.path.join(rgbd_dir, img_file)
        src_lbl = os.path.join(labels_dir, label_file)

        dst_img = os.path.join(base_path, split_name, 'images', img_file)
        dst_lbl = os.path.join(base_path, split_name, 'labels', label_file)

        if os.path.exists(src_lbl):  # Only copy if label exists
            shutil.copyfile(src_img, dst_img)
            shutil.copyfile(src_lbl, dst_lbl)
        else:
            print(f"⚠️ Label missing for: {img_file}")

# Perform copy
copy_files(train_files, 'train')
copy_files(val_files, 'val')
copy_files(test_files, 'test')

print("✅ Dataset split complete!")


Copying to test: 100%|██████████| 51/51 [00:25<00:00,  1.96it/s]

✅ Dataset split complete!


In [15]:
model.train(
    data='/content/drive/MyDrive/ColabNotebooks/data.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    project='rgbd_yolo_training',
    name='yolov8n_rgbd',
    exist_ok=True
)

Ultralytics 8.3.190 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ColabNotebooks/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_rgbd, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7eea6679bdd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [20]:
!cp -r /content/rgbd_yolo_training /content/drive/MyDrive/ColabNotebooks/

In [18]:
# Fix the first conv layer for 4-channel input (RGBD)
old_conv = model.model.model[0].conv  # Get the nn.Conv2d layer inside the custom Conv block
new_conv = nn.Conv2d(
    in_channels=4,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None
)

# Copy pretrained weights for RGB channels
with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight  # Copy RGB weights
    new_conv.weight[:, 3] = old_conv.weight[:, 0]  # Use R channel weights for D (or random init)

# Replace the conv layer
model.model.model[0].conv = new_conv


In [19]:
model.train(
    data='/content/drive/MyDrive/ColabNotebooks/data.yaml',
    epochs=150,
    imgsz=640,
    batch=8,
    lr0=0.001,
    weight_decay=0.001,
    device=0,
    name='yolov8_rgbd_scratch',
    pretrained=False
)


Ultralytics 8.3.190 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ColabNotebooks/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_rgbd_scratch, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, pl

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7eea06e4f560>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [21]:
!cp -r runs/detect/yolov8_rgbd_scratch /content/drive/MyDrive/ColabNotebooks/